In [32]:


from data_pipeline.data_procesing_pipeline import *

In [102]:
df = pd.read_excel("data/excel_files/2025_7-8.xlsx", sheet_name="DATA", header=5)
df = df[~df['EIC-код'].isin(['Grand Total', 'Загальний підсумок'])].copy()
df

,EIC-код,Група,Дата,Year,Month,Day,Hour,Sum of кВт,Average of Ціна розподілу ЕЕ грн. без ПДВ/кВт*год,Average of Ціна ЕЕ грн. без ПДВ/кВт*год
0,62Z0008583037334,А,2025-07-01,2025.0,7.0,1.0,1.0,19.000000,2.37385,5.441006
1,62Z0008583037334,А,2025-07-01,2025.0,7.0,1.0,2.0,16.000000,2.37385,5.441006
2,62Z0008583037334,А,2025-07-01,2025.0,7.0,1.0,3.0,16.000000,2.37385,5.441006
3,62Z0008583037334,А,2025-07-01,2025.0,7.0,1.0,4.0,15.000000,2.37385,5.441006
4,62Z0008583037334,А,2025-07-01,2025.0,7.0,1.0,5.0,14.000000,2.37385,5.441006
...,...,...,...,...,...,...,...,...,...,...
589987,62Z9997819406173,Б,2025-08-31,2025.0,8.0,31.0,20.0,22.540932,1.76339,5.447321
589988,62Z9997819406173,Б,2025-08-31,2025.0,8.0,31.0,21.0,24.207382,1.76339,5.447321
589989,62Z9997819406173,Б,2025-08-31,2025.0,8.0,31.0,22.0,25.211150,1.76339,5.447321
589990,62Z9997819406173,Б,2025-08-31,2025.0,8.0,31.0,23.0,22.579913,1.76339,5.447321


In [105]:
df = df_original.copy()

In [106]:
# df = df[df['EIC-код'] == df['EIC-код'][0]].copy()

df['Дата'] = pd.to_datetime(df['Дата'], errors='coerce')
df['Дата'] = df['Дата'] + pd.to_timedelta(df['Hour'].fillna(0), unit='h')

df = df[['EIC-код', 'Дата', 'Sum of кВт']]

In [107]:
# df.to_excel("data/inference/all_stations_sample.xlsx", index=False)

In [98]:
import pandas as pd


def extend_datetime_range(
    df: pd.DataFrame,
    start_dt,
    end_dt,
    eic_col: str = "EIC-код",
    dt_col: str = "Дата",
    value_col: str = "Sum of кВт",
    freq: str = "H",
) -> pd.DataFrame:
    """
    Extends the datetime range for each unique EIC code.

    Existing values are preserved.
    Future timestamps are added with NaN in the value column.

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe.
    start_dt : str or datetime-like
        Start of the new datetime range.
        Must be >= current minimum timestamp.
    end_dt : str or datetime-like
        End of the new datetime range.
    eic_col : str
        Name of EIC code column.
    dt_col : str
        Name of datetime column.
    value_col : str
        Name of value column.
    freq : str
        Frequency of timestamps (default hourly).

    Returns
    -------
    pd.DataFrame
    """

    df = df.copy()

    # Ensure datetime
    df[dt_col] = pd.to_datetime(df[dt_col])

    start_dt = pd.to_datetime(start_dt)
    end_dt = pd.to_datetime(end_dt)

    current_min = df[dt_col].min()

    if start_dt < current_min:
        raise ValueError(
            f"start_dt ({start_dt}) cannot be earlier than "
            f"existing minimum timestamp ({current_min})"
        )

    # Create full datetime range
    full_dates = pd.date_range(start=start_dt, end=end_dt, freq=freq)

    # Unique EICs
    unique_eics = df[eic_col].unique()

    # Full cartesian product
    full_index = pd.MultiIndex.from_product(
        [unique_eics, full_dates],
        names=[eic_col, dt_col]
    )

    full_df = pd.DataFrame(index=full_index).reset_index()

    # Merge with original data
    result = full_df.merge(
        df,
        on=[eic_col, dt_col],
        how="left"
    )

    # Sort
    result = result.sort_values([eic_col, dt_col]).reset_index(drop=True)

    return result

In [99]:
extended_df = extend_datetime_range(
    df,
    start_dt="2025-09-01",
    end_dt="2025-10-30"
)
extended_df = pd.concat([df, extended_df]).copy()
extended_df

/var/folders/5w/bchvy66s6vbgncfkws1skvf40000gp/T/ipykernel_16047/205549039.py:59: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  full_dates = pd.date_range(start=start_dt, end=end_dt, freq=freq)


,EIC-код,Дата,Sum of кВт
0,62Z0008583037334,2025-07-01 01:00:00,19.0
1,62Z0008583037334,2025-07-01 02:00:00,16.0
2,62Z0008583037334,2025-07-01 03:00:00,16.0
3,62Z0008583037334,2025-07-01 04:00:00,15.0
4,62Z0008583037334,2025-07-01 05:00:00,14.0
...,...,...,...
1412,62Z0008583037334,2025-10-29 20:00:00,NaN
1413,62Z0008583037334,2025-10-29 21:00:00,NaN
1414,62Z0008583037334,2025-10-29 22:00:00,NaN
1415,62Z0008583037334,2025-10-29 23:00:00,NaN


In [100]:
df = extended_df.copy()

In [101]:
static_path = "../data/raw/8month2025.xlsx"
static_data_dict = get_static_data_dict(path=static_path)

df = df.merge(static_data_dict, on='EIC-код', how='left', suffixes=('', '_dict'))

df = add_solar(df)

df = add_max_power(df)

# df = drop_invalid_data(df)

cols_to_leave = ['EIC-код',
                 'Дата', 'Sum of кВт',
                 'Тип',
                 'Область', 'GPS-координати - Широта', 'GPS-координати - Довгота', 'ОСР опис', 'Адреса',
                 'max_power', 'max_solar']

rename_cols = {
    'Дата': 'datetime',
    'Sum of кВт': 'sum_of_kWh',
    'EIC-код': 'eic_code',
    'Тип': 'station_type',
    'Область': 'oblast',
    'GPS-координати - Широта': 'latitude',
    'GPS-координати - Довгота': 'longitude',
    'ОСР опис': 'osr_desc',
    'Адреса': 'address'
}

df = df[cols_to_leave].copy()
df = df.rename(columns=rename_cols)

# df = keep_stations_with_90_days(df)

df = add_weather_wrapper(df, no_weather_path, weather_path, checkpoint_path, weather_cols_to_create_all)

df['datetime'] = pd.to_datetime(df['datetime'])

df = add_ev_power(df, static_path, "../data/additional_data/EV_chargers.xlsx")

df = add_calendar_features(df)

df = build_time_index(df)

df = df[[
    'eic_code', 'datetime', 'sum_of_kWh',
    'max_power', 'max_solar', 'max_ev_power_kW',
    'latitude', 'longitude', 'osr_desc', 'station_type', 'oblast']
    +
    weather_cols_to_create_all
]

cols_to_rename = {
    'max_power': 'max_power',
    'max_solar': 'max_solar',
    'max_ev_power_kW': 'max_ev',
    'osr_desc': 'dso_desc',
}

df = df.rename(columns=cols_to_rename)

# df = trimm_stations(df)

df = add_prices(df)

static_cols = [
    'latitude', 'longitude', 'eic_code', 'dso_desc', 'station_type', 'oblast'
]

calendar_cols = [
    'Month', 'Day', 'Hour', 'day_of_week', 'season'
]

GLOBAL_MIN_DT = df["datetime"].min()

df = df.sort_values(['eic_code', 'datetime'])

df['time_idx'] = (
        (df['datetime'] - GLOBAL_MIN_DT)
        .dt.total_seconds() // 3600
).astype(int)

print(f"Number of missing timesteps: {len(find_missing_timesteps(df))}")

df

/Users/levkupybida/PycharmProjects/Diploma/data_procesing_pipeline.py:53: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(static_data_dict["EIC-код"].map(lat_map))
/Users/levkupybida/PycharmProjects/Diploma/data_procesing_pipeline.py:59: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(static_data_dict["EIC-код"].map(lon_map))


Gas stations with weather: 0
Gas stations with no weather: 1


Gas stations:   0%|          | 0/1 [00:00<?, ?it/s]

Number of missing timesteps: 1


,eic_code,datetime,sum_of_kWh,max_power,max_solar,max_ev,latitude,longitude,dso_desc,station_type,...,wind_direction_10m,wind_gusts_10m,shortwave_radiation,diffuse_radiation,direct_normal_irradiance,sell_bm_price,buy_bm_price,dam_price,data_subset,time_idx
0,62Z0008583037334,2025-07-01 01:00:00+03:00,19.0,110.0,25.52,0.0,48.44243,22.19219,Ужгород,ОККО-комплекс,...,356.0,22.3,0.0,0.0,0.0,0.01,5846.84,5568.42,val,0
1,62Z0008583037334,2025-07-01 02:00:00+03:00,16.0,110.0,25.52,0.0,48.44243,22.19219,Ужгород,ОККО-комплекс,...,13.0,24.1,0.0,0.0,0.0,0.01,5449.50,5190.00,val,1
2,62Z0008583037334,2025-07-01 03:00:00+03:00,16.0,110.0,25.52,0.0,48.44243,22.19219,Ужгород,ОККО-комплекс,...,358.0,17.3,0.0,0.0,0.0,0.01,5132.40,4888.00,val,2
3,62Z0008583037334,2025-07-01 04:00:00+03:00,15.0,110.0,25.52,0.0,48.44243,22.19219,Ужгород,ОККО-комплекс,...,330.0,15.1,0.0,0.0,0.0,0.01,4513.95,4299.00,val,3
4,62Z0008583037334,2025-07-01 05:00:00+03:00,14.0,110.0,25.52,0.0,48.44243,22.19219,Ужгород,ОККО-комплекс,...,340.0,16.2,0.0,0.0,0.0,0.01,5132.40,4888.00,val,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2900,62Z0008583037334,2025-10-29 20:00:00+02:00,NaN,110.0,25.52,0.0,48.44243,22.19219,Ужгород,ОККО-комплекс,...,192.0,23.0,0.0,0.0,0.0,14156.65,16590.00,14901.74,test,2900
2901,62Z0008583037334,2025-10-29 21:00:00+02:00,NaN,110.0,25.52,0.0,48.44243,22.19219,Ужгород,ОККО-комплекс,...,176.0,19.4,0.0,0.0,0.0,14160.38,16800.00,14905.66,test,2901
2902,62Z0008583037334,2025-10-29 22:00:00+02:00,NaN,110.0,25.52,0.0,48.44243,22.19219,Ужгород,ОККО-комплекс,...,153.0,20.2,0.0,0.0,0.0,12350.00,16800.00,13000.00,test,2902
2903,62Z0008583037334,2025-10-29 23:00:00+02:00,NaN,110.0,25.52,0.0,48.44243,22.19219,Ужгород,ОККО-комплекс,...,149.0,22.0,0.0,0.0,0.0,5043.55,8662.50,5309.00,test,2903
